# Silver model backtest

Load `silver.nba_player_gamelogs`, engineer features, run saved quantile models on chosen dates, and compare predictions vs actuals.

**Configure** `TEST_DATES` and `PROPS` in the config cell, then run all cells.

In [1]:
import warnings
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd

project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src.pipeline.gold import _read_silver_slice
from src.pipeline.features.min_features import MIN_FEATURES, build_min_dataset
from src.pipeline.features.ppm_features import PPM_FEATURES, build_ppm_dataset
from src.pipeline.features.rpm_features import RPM_FEATURES, build_rpm_dataset
from src.pipeline.features.apm_features import APM_FEATURES, build_apm_dataset
from src.utils.ml import load_model_bundle, predict_quantiles

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## Config

In [2]:
# Dates to evaluate (YYYY-MM-DD). Model features on each row are leakage-safe
# (built from prior games only), then compared to that night's actuals.
TEST_DATES = [
    "2026-04-20",
    # "2026-04-21",
]

# Props to score. Each needs a matching *.joblib under src/models/saved_models/
PROPS = ["min", "ppm", "rpm", "apm"]

# Optional filters
MIN_MINUTES = 10          # drop rows with actual MIN below this
PLAYER_NAME = None        # e.g. "Jalen Brunson" or None for all
TEAM_ABBREVIATION = None  # e.g. "NYK" or None for all

# Seasons needed to build features (history + test nights)
SEASONS = [
    ("2021-22", "S22"),
    ("2022-23", "S23"),
    ("2023-24", "S24"),
    ("2024-25", "S25"),
    ("2025-26", "S26"),
]
SEASON_TYPES = ("Regular Season", "Playoffs")

MODELS_DIR = Path("src/models/saved_models")
CACHE_DIR = Path("data/cache")
FORCE_REBUILD = False  # True after changing feature modules

BUILDERS = {
    "min": build_min_dataset,
    "ppm": build_ppm_dataset,
    "rpm": build_rpm_dataset,
    "apm": build_apm_dataset,
}

# Actual counting-stat column for rate props (pred_stat = q50 * actual MIN)
COUNT_STAT = {
    "min": "MIN",
    "ppm": "PTS",
    "rpm": "REB",
    "apm": "AST",
}
TARGET_COL = {
    "min": "MIN",
    "ppm": "PTS_PER_MIN",
    "rpm": "REB_PER_MIN",
    "apm": "AST_PER_MIN",
}

print("Test dates:", TEST_DATES)
print("Props:", PROPS)

Test dates: ['2026-04-20']
Props: ['min', 'ppm', 'rpm', 'apm']


## Load silver + engineer features (cached per prop)

In [3]:
def load_season_from_silver(season_year: str) -> pd.DataFrame:
    parts = []
    for season_type in SEASON_TYPES:
        part = _read_silver_slice(season_year, season_type, league="nba")
        print(f"  {season_year} {season_type}: {len(part):,} rows")
        if not part.empty:
            parts.append(part)
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True).sort_values("GAME_DATE")


PROP_FEATURE_LISTS = {
    "min": MIN_FEATURES,
    "ppm": PPM_FEATURES,
    "rpm": RPM_FEATURES,
    "apm": APM_FEATURES,
}


def load_or_build_prop_frame(prop: str) -> pd.DataFrame:
    cache_path = CACHE_DIR / f"{prop}_from_silver.parquet"
    target = TARGET_COL[prop]

    # Prefer the saved model's feature list so cache matches what we'll predict with.
    try:
        needed = list(load_model_bundle(prop, models_dir=MODELS_DIR)["feature_names"])
    except Exception:
        needed = list(PROP_FEATURE_LISTS[prop])

    df = None
    if cache_path.exists() and not FORCE_REBUILD:
        cached = pd.read_parquet(cache_path)
        missing = [c for c in needed if c not in cached.columns]
        if missing:
            print(
                f"[{prop}] cache stale — missing {len(missing)} features "
                f"(e.g. {missing[:3]}); rebuilding…"
            )
        else:
            df = cached
            df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
            print(f"[{prop}] cache {cache_path.name}: {len(df):,} rows")

    if df is None:
        season_dfs, season_map = [], {}
        for season_year, label in SEASONS:
            part = load_season_from_silver(season_year)
            if part.empty:
                print(f"  ⚠ skip {season_year}: no silver rows")
                continue
            season_map[len(season_dfs)] = label
            season_dfs.append(part)
            print(f"  → {label}: {len(part):,} raw rows")

        if not season_dfs:
            raise RuntimeError("No silver seasons loaded — run fetch_raw.py first")

        print(f"[{prop}] building features…")
        df = BUILDERS[prop](season_dfs, season_map=season_map)
        df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
        df = df.sort_values("GAME_DATE").reset_index(drop=True)

        # Ensure counting stats exist for rate → volume conversion
        if (
            target != "MIN"
            and target not in df.columns
            and {COUNT_STAT[prop], "MIN"}.issubset(df.columns)
        ):
            df[target] = df[COUNT_STAT[prop]] / df["MIN"].replace(0, np.nan)

        still_missing = [c for c in needed if c not in df.columns]
        if still_missing:
            raise KeyError(
                f"[{prop}] engineered frame still missing model features: {still_missing}"
            )

        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        df.to_parquet(cache_path, index=False)
        print(f"[{prop}] wrote {cache_path}: {len(df):,} rows")

    return df


prop_frames = {prop: load_or_build_prop_frame(prop) for prop in PROPS}
for prop, df in prop_frames.items():
    print(prop, df.shape, "date range",
          df["GAME_DATE"].min().date(), "→", df["GAME_DATE"].max().date())


[min] cache min_from_silver.parquet: 127,994 rows
[ppm] cache ppm_from_silver.parquet: 118,434 rows
[rpm] cache rpm_from_silver.parquet: 118,434 rows
[apm] cache apm_from_silver.parquet: 105,620 rows
min (127994, 228) date range 2021-10-19 → 2026-06-13
ppm (118434, 256) date range 2021-10-19 → 2026-06-13
rpm (118434, 234) date range 2021-10-19 → 2026-06-13
apm (105620, 250) date range 2021-10-19 → 2026-06-13


## Predict vs actuals for configured dates

In [4]:
def slice_test_rows(df: pd.DataFrame, dates: list[str]) -> pd.DataFrame:
    dates_ts = pd.to_datetime(dates)
    out = df[df["GAME_DATE"].dt.normalize().isin(dates_ts.normalize())].copy()
    if PLAYER_NAME:
        out = out[out["PLAYER_NAME"] == PLAYER_NAME]
    if TEAM_ABBREVIATION and "TEAM_ABBREVIATION" in out.columns:
        out = out[out["TEAM_ABBREVIATION"] == TEAM_ABBREVIATION]
    if "MIN" in out.columns and MIN_MINUTES is not None:
        out = out[out["MIN"] >= MIN_MINUTES]
    return out.sort_values(["GAME_DATE", "PLAYER_NAME"]).reset_index(drop=True)


def score_prop_on_dates(prop: str, feat_df: pd.DataFrame) -> pd.DataFrame:
    test = slice_test_rows(feat_df, TEST_DATES)
    if test.empty:
        print(f"[{prop}] no rows for {TEST_DATES}")
        return pd.DataFrame()

    bundle = load_model_bundle(prop, models_dir=MODELS_DIR)
    print(f"[{prop}] model={Path(bundle['model_path']).name}  rows={len(test):,}")

    # Diagnose incomplete rows before predict_quantiles raises
    feat_names = bundle["feature_names"]
    from src.utils.ml import align_feature_frame
    X_check = align_feature_frame(test, feat_names)
    complete = X_check.notna().all(axis=1)
    if not complete.any():
        nan_rate = X_check.isna().mean().sort_values(ascending=False)
        raise ValueError(
            f"[{prop}] no complete feature rows for {TEST_DATES}. "
            f"Worst NaN features:\n{nan_rate.head(10).to_string()}"
        )
    if complete.sum() < len(test):
        print(
            f"[{prop}] dropping {int((~complete).sum())} / {len(test)} rows "
            f"with incomplete features"
        )
        test = test.loc[complete].reset_index(drop=True)

    preds = predict_quantiles(prop, test, models_dir=MODELS_DIR)
    keys = ["GAME_ID", "PLAYER_ID"]
    target = TARGET_COL[prop]
    count_col = COUNT_STAT[prop]

    meta = ["GAME_DATE", "PLAYER_NAME", "TEAM_ABBREVIATION", "MATCHUP", "STARTING"]
    need = list(dict.fromkeys(keys + meta + ["MIN", target, count_col]))
    missing = [c for c in need if c not in test.columns and c not in keys]
    # only require keys + target; others optional
    if target not in test.columns:
        raise KeyError(
            f"[{prop}] engineered frame missing target {target!r}. "
            f"Columns sample: {list(test.columns)[:30]}"
        )

    actuals = test[[c for c in need if c in test.columns]].drop_duplicates(keys).copy()
    actuals["_actual_rate"] = pd.to_numeric(actuals[target], errors="coerce")
    actuals["_actual_min"] = (
        pd.to_numeric(actuals["MIN"], errors="coerce")
        if "MIN" in actuals.columns
        else actuals["_actual_rate"]
    )
    if count_col in actuals.columns:
        actuals["_actual_stat"] = pd.to_numeric(actuals[count_col], errors="coerce")
    else:
        actuals["_actual_stat"] = actuals["_actual_rate"]

    # Drop raw target/count/min from actuals side to avoid merge name clashes;
    # keep identity + display meta.
    keep_actuals = (
        keys
        + [c for c in meta if c in actuals.columns]
        + ["_actual_rate", "_actual_min", "_actual_stat"]
    )
    actuals = actuals[keep_actuals]

    drop_from_preds = [
        c for c in ("GAME_DATE", "PLAYER_NAME")
        if c in preds.columns and c in actuals.columns
    ]
    preds_clean = preds.drop(columns=drop_from_preds, errors="ignore")
    merged = preds_clean.merge(actuals, on=keys, how="left")

    merged["ACTUAL_RATE"] = pd.to_numeric(merged["_actual_rate"], errors="coerce")
    merged["PRED_P10"] = pd.to_numeric(merged["Q_0.10"], errors="coerce")
    merged["PRED_P50"] = pd.to_numeric(merged["Q_0.50"], errors="coerce")
    merged["PRED_P90"] = pd.to_numeric(merged["Q_0.90"], errors="coerce")
    merged["ERR_P50"] = merged["PRED_P50"] - merged["ACTUAL_RATE"]
    merged["ABS_ERR"] = merged["ERR_P50"].abs()
    merged["IN_80"] = (
        (merged["ACTUAL_RATE"] >= merged["PRED_P10"])
        & (merged["ACTUAL_RATE"] <= merged["PRED_P90"])
    )

    mins = pd.to_numeric(merged["_actual_min"], errors="coerce")
    if prop == "min":
        merged["ACTUAL_STAT"] = merged["ACTUAL_RATE"]
        merged["PRED_STAT_P50"] = merged["PRED_P50"]
        merged["PRED_STAT_P10"] = merged["PRED_P10"]
        merged["PRED_STAT_P90"] = merged["PRED_P90"]
    else:
        merged["ACTUAL_STAT"] = pd.to_numeric(merged["_actual_stat"], errors="coerce")
        merged["PRED_STAT_P50"] = merged["PRED_P50"] * mins
        merged["PRED_STAT_P10"] = merged["PRED_P10"] * mins
        merged["PRED_STAT_P90"] = merged["PRED_P90"] * mins

    merged["MIN"] = mins
    merged["STAT_ABS_ERR"] = (merged["PRED_STAT_P50"] - merged["ACTUAL_STAT"]).abs()
    merged["PROP"] = prop
    return merged


results = []
for prop in PROPS:
    scored = score_prop_on_dates(prop, prop_frames[prop])
    if not scored.empty:
        results.append(scored)

if not results:
    raise RuntimeError("No scored rows — check TEST_DATES / silver coverage / models")

backtest = pd.concat(results, ignore_index=True)
print(f"\nBacktest rows: {len(backtest):,}")
backtest.head()


[min] model=min_quantile_xgb_v2_2026-05-11.joblib  rows=46
[ppm] model=ppm_quantile_xgb_2026-06-13.joblib  rows=46
[rpm] model=rpm_quantile_xgb_2026-06-13.joblib  rows=46
[apm] model=apm_quantile_xgb_2026-06-13.joblib  rows=40

Backtest rows: 178


,GAME_ID,PLAYER_ID,Q_0.10,Q_0.50,Q_0.90,PREDICTION,PROP,MODEL_PATH,MODEL_ID,PREDICTED_AT,GAME_DATE,PLAYER_NAME,TEAM_ABBREVIATION,MATCHUP,STARTING,_actual_rate,_actual_min,_actual_stat,ACTUAL_RATE,PRED_P10,PRED_P50,PRED_P90,ERR_P50,ABS_ERR,IN_80,ACTUAL_STAT,PRED_STAT_P50,PRED_STAT_P10,PRED_STAT_P90,MIN,STAT_ABS_ERR
0,0042500162,203932,22.270609,29.227121,36.005070,29.227121,min,src\models\saved_models\min_quantile_xgb_v2_20...,None,2026-07-12 22:24:18.864413+00:00,2026-04-20,Aaron Gordon,DEN,DEN vs. MIN,1,36.883333,36.883333,36.883333,36.883333,22.270609,29.227121,36.005070,-7.656212,7.656212,False,36.883333,29.227121,22.270609,36.005070,36.883333,7.656212
1,0042500162,1630162,27.617235,34.889519,39.227951,34.889519,min,src\models\saved_models\min_quantile_xgb_v2_20...,None,2026-07-12 22:24:18.864413+00:00,2026-04-20,Anthony Edwards,MIN,MIN @ DEN,1,40.033333,40.033333,40.033333,40.033333,27.617235,34.889519,39.227951,-5.143815,5.143815,False,40.033333,34.889519,27.617235,39.227951,40.033333,5.143815
2,0042500162,1630245,17.840633,24.874493,30.651817,24.874493,min,src\models\saved_models\min_quantile_xgb_v2_20...,None,2026-07-12 22:24:18.864413+00:00,2026-04-20,Ayo Dosunmu,MIN,MIN @ DEN,0,22.425000,22.425000,22.425000,22.425000,17.840633,24.874493,30.651817,2.449493,2.449493,True,22.425000,24.874493,17.840633,30.651817,22.425000,2.449493
3,0042500132,1627742,26.379021,35.570744,39.609287,35.570744,min,src\models\saved_models\min_quantile_xgb_v2_20...,None,2026-07-12 22:24:18.864413+00:00,2026-04-20,Brandon Ingram,TOR,TOR @ CLE,1,35.895000,35.895000,35.895000,35.895000,26.379021,35.570744,39.609287,-0.324256,0.324256,True,35.895000,35.570744,26.379021,39.609287,35.895000,0.324256
4,0042500162,1628971,14.490190,20.842039,27.708471,20.842039,min,src\models\saved_models\min_quantile_xgb_v2_20...,None,2026-07-12 22:24:18.864413+00:00,2026-04-20,Bruce Brown,DEN,DEN vs. MIN,0,16.460000,16.460000,16.460000,16.460000,14.490190,20.842039,27.708471,4.382039,4.382039,True,16.460000,20.842039,14.490190,27.708471,16.460000,4.382039


## Summary metrics

In [5]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (prop, day), g in df.groupby(["PROP", "GAME_DATE"]):
        rows.append({
            "PROP": prop,
            "GAME_DATE": pd.to_datetime(day).date(),
            "n": len(g),
            "mae_rate": g["ABS_ERR"].mean(),
            "mae_stat": g["STAT_ABS_ERR"].mean(),
            "coverage_80": g["IN_80"].mean(),
            "bias_p50": g["ERR_P50"].mean(),
        })
    # Overall per prop
    for prop, g in df.groupby("PROP"):
        rows.append({
            "PROP": prop,
            "GAME_DATE": "ALL",
            "n": len(g),
            "mae_rate": g["ABS_ERR"].mean(),
            "mae_stat": g["STAT_ABS_ERR"].mean(),
            "coverage_80": g["IN_80"].mean(),
            "bias_p50": g["ERR_P50"].mean(),
        })
    out = pd.DataFrame(rows)
    return out.sort_values(["PROP", "GAME_DATE"]).reset_index(drop=True)


summary = summarize(backtest)
summary.style.format({
    "mae_rate": "{:.4f}",
    "mae_stat": "{:.2f}",
    "coverage_80": "{:.1%}",
    "bias_p50": "{:+.4f}",
})

,PROP,GAME_DATE,n,mae_rate,mae_stat,coverage_80,bias_p50
0,apm,2026-04-20,40,0.0416,1.35,87.5%,+0.0132
1,apm,ALL,40,0.0416,1.35,87.5%,+0.0132
2,min,2026-04-20,46,3.9149,3.91,82.6%,-1.5354
3,min,ALL,46,3.9149,3.91,82.6%,-1.5354
4,ppm,2026-04-20,46,0.1492,4.22,82.6%,+0.0301
5,ppm,ALL,46,0.1492,4.22,82.6%,+0.0301
6,rpm,2026-04-20,46,0.0702,1.90,80.4%,-0.0063
7,rpm,ALL,46,0.0702,1.90,80.4%,-0.0063


## Showcase: MIN, then MIN × rate props

Volume props use **predicted minutes × predicted per-minute rate**:

| Showcase | Formula | Actual |
|----------|---------|--------|
| MIN | `PRED_P50` minutes | `MIN` |
| MIN × PPM | `min_p50 × ppm_p50` → PTS | `PTS` |
| MIN × APM | `min_p50 × apm_p50` → AST | `AST` |
| MIN × RPM | `min_p50 × rpm_p50` → REB | `REB` |

Bands use the matching quantile for both factors (`P10×P10`, `P50×P50`, `P90×P90`).


In [6]:
SHOWCASE_DATE = TEST_DATES[0]  # change to zoom a specific night


def _prop_slice(prop: str, date: str) -> pd.DataFrame:
    g = backtest[
        (backtest["PROP"] == prop)
        & (pd.to_datetime(backtest["GAME_DATE"]).dt.normalize()
           == pd.to_datetime(date).normalize())
    ].copy()
    return g


def _pick(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    return df[[c for c in cols if c in df.columns]].copy()


min_day = _prop_slice("min", SHOWCASE_DATE)
if min_day.empty:
    raise RuntimeError(
        f"No MIN rows for {SHOWCASE_DATE}. "
        "Include 'min' in PROPS and re-run the predict cell."
    )

# ── 1) Minutes ────────────────────────────────────────────────────────────────
min_show = _pick(min_day, [
    "PLAYER_NAME", "TEAM_ABBREVIATION", "MATCHUP", "STARTING",
    "PRED_P10", "PRED_P50", "PRED_P90", "ACTUAL_STAT", "STAT_ABS_ERR", "IN_80",
]).rename(columns={
    "PRED_P10": "MIN_P10",
    "PRED_P50": "MIN_P50",
    "PRED_P90": "MIN_P90",
    "ACTUAL_STAT": "ACTUAL_MIN",
    "STAT_ABS_ERR": "MIN_ABS_ERR",
    "IN_80": "MIN_IN_80",
}).sort_values("MIN_ABS_ERR", ascending=False)

print(f"=== MIN @ {SHOWCASE_DATE} — {len(min_show)} players ===")
print(
    f"MAE: {min_show['MIN_ABS_ERR'].mean():.2f} min | "
    f"80% coverage: {min_show['MIN_IN_80'].mean():.1%}"
)
display(min_show.head(25))


# ── 2) MIN × rate → counting stats ────────────────────────────────────────────
RATE_SPECS = [
    ("ppm", "PTS", "MIN×PPM → PTS"),
    ("apm", "AST", "MIN×APM → AST"),
    ("rpm", "REB", "MIN×RPM → REB"),
]

volume_frames = []
min_keys = min_day[["GAME_ID", "PLAYER_ID", "PLAYER_NAME", "TEAM_ABBREVIATION", "MATCHUP",
                    "STARTING", "PRED_P10", "PRED_P50", "PRED_P90", "ACTUAL_STAT"]].rename(
    columns={
        "PRED_P10": "MIN_P10",
        "PRED_P50": "MIN_P50",
        "PRED_P90": "MIN_P90",
        "ACTUAL_STAT": "ACTUAL_MIN",
    }
)

for prop, actual_col, label in RATE_SPECS:
    rate = _prop_slice(prop, SHOWCASE_DATE)
    if rate.empty:
        print(f"\n=== {label} — skipped (no '{prop}' rows) ===")
        continue

    r = rate[["GAME_ID", "PLAYER_ID", "PRED_P10", "PRED_P50", "PRED_P90",
              "ACTUAL_STAT", "ACTUAL_RATE"]].rename(columns={
        "PRED_P10": "RATE_P10",
        "PRED_P50": "RATE_P50",
        "PRED_P90": "RATE_P90",
        "ACTUAL_STAT": "ACTUAL_FROM_RATE",
        "ACTUAL_RATE": "ACTUAL_RATE",
    })
    # Prefer raw counting stat from rate frame if present via _actual_stat already in ACTUAL_STAT
    # Rebuild actual counting stat from silver-backed ACTUAL_STAT on rate prop
    # (for ppm/apm/rpm ACTUAL_STAT is PTS/REB/AST)
    merged = min_keys.merge(r, on=["GAME_ID", "PLAYER_ID"], how="inner")
    merged["PRED_P10"] = merged["MIN_P10"] * merged["RATE_P10"]
    merged["PRED_P50"] = merged["MIN_P50"] * merged["RATE_P50"]
    merged["PRED_P90"] = merged["MIN_P90"] * merged["RATE_P90"]
    merged["ACTUAL"] = merged["ACTUAL_FROM_RATE"]
    merged["ABS_ERR"] = (merged["PRED_P50"] - merged["ACTUAL"]).abs()
    merged["IN_80"] = (
        (merged["ACTUAL"] >= merged["PRED_P10"])
        & (merged["ACTUAL"] <= merged["PRED_P90"])
    )
    merged["SHOWCASE"] = label
    merged["STAT"] = actual_col

    print(f"\n=== {label} @ {SHOWCASE_DATE} — {len(merged)} players ===")
    print(
        f"MAE: {merged['ABS_ERR'].mean():.2f} {actual_col} | "
        f"80% coverage: {merged['IN_80'].mean():.1%}"
    )
    show = merged[[
        "PLAYER_NAME", "TEAM_ABBREVIATION", "MATCHUP",
        "MIN_P50", "ACTUAL_MIN", "RATE_P50", "ACTUAL_RATE",
        "PRED_P10", "PRED_P50", "PRED_P90", "ACTUAL", "ABS_ERR", "IN_80",
    ]].sort_values("ABS_ERR", ascending=False)
    display(show.head(25))
    volume_frames.append(merged)

# Combined leaderboard across volume props
if volume_frames:
    volume = pd.concat(volume_frames, ignore_index=True)
    print("\n=== Volume summary (MIN × rate) ===")
    vol_summary = (
        volume.groupby("SHOWCASE")
        .agg(n=("ABS_ERR", "size"), mae=("ABS_ERR", "mean"), coverage_80=("IN_80", "mean"))
        .reset_index()
    )
    display(vol_summary.style.format({"mae": "{:.2f}", "coverage_80": "{:.1%}"}))


=== MIN @ 2026-04-20 — 46 players ===
MAE: 3.91 min | 80% coverage: 82.6%


,PLAYER_NAME,TEAM_ABBREVIATION,MATCHUP,STARTING,MIN_P10,MIN_P50,MIN_P90,ACTUAL_MIN,MIN_ABS_ERR,MIN_IN_80
25,Jonathan Kuminga,ATL,ATL @ NYK,0,15.023561,22.997015,30.752520,34.733333,11.736318,False
21,Jamal Shead,TOR,TOR @ CLE,1,20.514738,28.639973,34.976036,37.600000,8.960027,False
13,Dyson Daniels,ATL,ATL @ NYK,1,25.041609,34.610722,38.480858,26.245000,8.365722,True
9,Dean Wade,CLE,CLE vs. TOR,1,14.808070,20.505333,27.157488,28.453333,7.948000,False
0,Aaron Gordon,DEN,DEN vs. MIN,1,22.270609,29.227121,36.005070,36.883333,7.656212,False
32,Miles McBride,NYK,NYK vs. ATL,0,12.501974,20.280464,28.200087,13.141667,7.138798,True
24,Jaylon Tyson,CLE,CLE vs. TOR,0,11.591450,18.339474,25.440586,11.733333,6.606140,True
30,Max Strus,CLE,CLE vs. TOR,0,14.309460,20.193312,25.622223,26.550000,6.356688,False
8,Collin Murray-Boyles,TOR,TOR @ CLE,0,10.520421,19.589935,27.220566,25.905000,6.315065,True
16,Ja'Kobe Walter,TOR,TOR @ CLE,0,13.402122,21.715137,29.600044,27.955000,6.239863,True



=== MIN×PPM → PTS @ 2026-04-20 — 46 players ===
MAE: 4.54 PTS | 80% coverage: 91.3%


,PLAYER_NAME,TEAM_ABBREVIATION,MATCHUP,MIN_P50,ACTUAL_MIN,RATE_P50,ACTUAL_RATE,PRED_P10,PRED_P50,PRED_P90,ACTUAL,ABS_ERR,IN_80
3,Brandon Ingram,TOR,TOR @ CLE,35.570744,35.895000,0.627928,0.195013,10.034424,22.335873,35.743484,7.0,15.335873,False
5,CJ McCollum,ATL,ATL @ NYK,31.162319,35.365000,0.597012,0.904849,9.002919,18.604284,32.155750,32.0,13.395716,True
35,Nickeil Alexander-Walker,ATL,ATL @ NYK,34.848534,38.166667,0.599912,0.235808,9.968578,20.906057,34.757942,9.0,11.906057,False
40,Rudy Gobert,MIN,MIN @ DEN,32.366146,27.623333,0.338440,0.072403,4.057560,10.954009,21.542992,2.0,8.954009,False
8,Collin Murray-Boyles,TOR,TOR @ CLE,19.589935,25.905000,0.436814,0.656244,2.026656,8.557161,18.597862,17.0,8.442839,True
33,Mitchell Robinson,NYK,NYK vs. ATL,17.239466,18.150000,0.284232,0.716253,0.892988,4.900010,13.497494,13.0,8.099990,True
32,Miles McBride,NYK,NYK vs. ATL,20.280464,13.141667,0.389091,0.000000,1.475149,7.890944,19.796635,0.0,7.890944,False
14,Evan Mobley,CLE,CLE vs. TOR,31.015678,33.416667,0.555851,0.748130,8.219638,17.240108,29.372519,25.0,7.759892,True
25,Jonathan Kuminga,ATL,ATL @ NYK,22.997015,34.733333,0.503368,0.547025,3.535020,11.575968,23.993704,19.0,7.424032,True
0,Aaron Gordon,DEN,DEN vs. MIN,29.227121,36.883333,0.513590,0.216900,5.472329,15.010771,27.345160,8.0,7.010771,True



=== MIN×APM → AST @ 2026-04-20 — 40 players ===
MAE: 1.28 AST | 80% coverage: 95.0%


,PLAYER_NAME,TEAM_ABBREVIATION,MATCHUP,MIN_P50,ACTUAL_MIN,RATE_P50,ACTUAL_RATE,PRED_P10,PRED_P50,PRED_P90,ACTUAL,ABS_ERR,IN_80
12,Dyson Daniels,ATL,ATL @ NYK,34.610722,26.245000,0.169259,0.076205,2.116750,5.858181,10.574809,2.0,3.858181,False
17,Jalen Johnson,ATL,ATL @ NYK,36.611259,35.633333,0.173362,0.084191,2.488616,6.347003,11.704768,3.0,3.347003,True
20,James Harden,CLE,CLE vs. TOR,36.131115,34.883333,0.199551,0.114668,2.955684,7.209991,12.580393,4.0,3.209991,True
7,Christian Braun,DEN,DEN vs. MIN,32.029179,35.350000,0.070005,0.141443,0.078362,2.242201,5.579395,5.0,2.757799,True
30,Nickeil Alexander-Walker,ATL,ATL @ NYK,34.848534,38.166667,0.098461,0.157205,0.999003,3.431227,7.104644,6.0,2.568773,True
11,Donte DiVincenzo,MIN,MIN @ DEN,30.148401,30.568333,0.115438,0.196282,0.777301,3.480264,7.455155,6.0,2.519736,True
31,Nikola Jokić,DEN,DEN vs. MIN,36.272892,40.261667,0.288235,0.198700,5.120596,10.455101,16.753201,8.0,2.455101,True
1,Anthony Edwards,MIN,MIN @ DEN,34.889519,40.033333,0.127438,0.049958,1.391553,4.446252,8.481692,2.0,2.446252,True
38,Scottie Barnes,TOR,TOR @ CLE,35.296356,39.650000,0.207903,0.126103,3.016675,7.338209,13.046316,5.0,2.338209,True
23,Josh Hart,NYK,NYK vs. ATL,31.617813,35.416667,0.119711,0.169412,1.109748,3.784991,8.327075,6.0,2.215009,True



=== MIN×RPM → REB @ 2026-04-20 — 46 players ===
MAE: 2.10 REB | 80% coverage: 89.1%


,PLAYER_NAME,TEAM_ABBREVIATION,MATCHUP,MIN_P50,ACTUAL_MIN,RATE_P50,ACTUAL_RATE,PRED_P10,PRED_P50,PRED_P90,ACTUAL,ABS_ERR,IN_80
27,Josh Hart,NYK,NYK vs. ATL,31.617813,35.416667,0.197743,0.367059,2.372338,6.252215,11.843371,13.0,6.747785,False
23,Jarrett Allen,CLE,CLE vs. TOR,28.002548,25.161667,0.309557,0.119229,3.556010,8.668375,15.387361,3.0,5.668375,False
1,Anthony Edwards,MIN,MIN @ DEN,34.889519,40.033333,0.131425,0.249792,1.500620,4.585348,8.669144,10.0,5.414652,False
42,Sandro Mamukelashvili,TOR,TOR @ CLE,21.208355,20.511667,0.230913,0.487527,1.499253,4.897291,11.246163,10.0,5.102709,True
39,RJ Barrett,TOR,TOR @ CLE,32.229156,37.716667,0.139520,0.238621,1.312907,4.496626,8.839435,9.0,4.503374,False
40,Rudy Gobert,MIN,MIN @ DEN,32.366146,27.623333,0.342597,0.253409,4.645302,11.088539,18.598528,7.0,4.088539,True
34,Naz Reid,MIN,MIN @ DEN,21.883575,27.033333,0.226100,0.332922,1.724970,4.947876,10.751509,9.0,4.052124,True
29,Karl-Anthony Towns,NYK,NYK vs. ATL,33.022667,33.775000,0.356027,0.236862,5.321987,11.756959,19.433241,8.0,3.756959,True
12,Donte DiVincenzo,MIN,MIN @ DEN,30.148401,30.568333,0.108866,0.228995,0.750559,3.282138,7.359857,7.0,3.717862,True
26,Jordan Clarkson,NYK,NYK vs. ATL,13.816890,11.083333,0.120415,0.451128,0.130143,1.663761,5.667507,5.0,3.336239,True



=== Volume summary (MIN × rate) ===


,SHOWCASE,n,mae,coverage_80
0,MIN×APM → AST,40,1.28,95.0%
1,MIN×PPM → PTS,46,4.54,91.3%
2,MIN×RPM → REB,46,2.10,89.1%


## Player-level detail

Set `DETAIL_PROP` / `DETAIL_DATE` to zoom in.

In [10]:
DETAIL_PROP = PROPS[0]
DETAIL_DATE = TEST_DATES[0]

detail = backtest[
    (backtest["PROP"] == DETAIL_PROP)
    & (pd.to_datetime(backtest["GAME_DATE"]).dt.normalize()
       == pd.to_datetime(DETAIL_DATE).normalize())
].copy()

show_cols = [
    c for c in [
        "PLAYER_NAME", "TEAM_ABBREVIATION", "MATCHUP", "MIN", "STARTING",
        "ACTUAL_RATE", "PRED_P10", "PRED_P50", "PRED_P90", "ABS_ERR", "IN_80",
        "ACTUAL_STAT", "PRED_STAT_P10", "PRED_STAT_P50", "PRED_STAT_P90", "STAT_ABS_ERR",
    ]
    if c in detail.columns
]

detail = detail.sort_values("STAT_ABS_ERR", ascending=False)
print(f"{DETAIL_PROP.upper()} @ {DETAIL_DATE} — {len(detail)} players")
detail[show_cols].head(40)

MIN @ 2026-04-20 — 46 players


,PLAYER_NAME,TEAM_ABBREVIATION,MATCHUP,MIN,STARTING,ACTUAL_RATE,PRED_P10,PRED_P50,PRED_P90,ABS_ERR,IN_80,ACTUAL_STAT,PRED_STAT_P10,PRED_STAT_P50,PRED_STAT_P90,STAT_ABS_ERR
25,Jonathan Kuminga,ATL,ATL @ NYK,34.733333,0,34.733333,15.023561,22.997015,30.752520,11.736318,False,34.733333,15.023561,22.997015,30.752520,11.736318
21,Jamal Shead,TOR,TOR @ CLE,37.600000,1,37.600000,20.514738,28.639973,34.976036,8.960027,False,37.600000,20.514738,28.639973,34.976036,8.960027
13,Dyson Daniels,ATL,ATL @ NYK,26.245000,1,26.245000,25.041609,34.610722,38.480858,8.365722,True,26.245000,25.041609,34.610722,38.480858,8.365722
9,Dean Wade,CLE,CLE vs. TOR,28.453333,1,28.453333,14.808070,20.505333,27.157488,7.948000,False,28.453333,14.808070,20.505333,27.157488,7.948000
0,Aaron Gordon,DEN,DEN vs. MIN,36.883333,1,36.883333,22.270609,29.227121,36.005070,7.656212,False,36.883333,22.270609,29.227121,36.005070,7.656212
32,Miles McBride,NYK,NYK vs. ATL,13.141667,0,13.141667,12.501974,20.280464,28.200087,7.138798,True,13.141667,12.501974,20.280464,28.200087,7.138798
24,Jaylon Tyson,CLE,CLE vs. TOR,11.733333,0,11.733333,11.591450,18.339474,25.440586,6.606140,True,11.733333,11.591450,18.339474,25.440586,6.606140
30,Max Strus,CLE,CLE vs. TOR,26.550000,0,26.550000,14.309460,20.193312,25.622223,6.356688,False,26.550000,14.309460,20.193312,25.622223,6.356688
8,Collin Murray-Boyles,TOR,TOR @ CLE,25.905000,0,25.905000,10.520421,19.589935,27.220566,6.315065,True,25.905000,10.520421,19.589935,27.220566,6.315065
16,Ja'Kobe Walter,TOR,TOR @ CLE,27.955000,0,27.955000,13.402122,21.715137,29.600044,6.239863,True,27.955000,13.402122,21.715137,29.600044,6.239863


In [9]:
# Biggest misses / hits for the detail slice
print("Worst misses (by counting-stat abs error)")
display(detail[show_cols].head(10))

print("\nBest hits")
display(detail.sort_values("STAT_ABS_ERR").head(10)[show_cols])

Worst misses (by counting-stat abs error)


,PLAYER_NAME,TEAM_ABBREVIATION,MATCHUP,MIN,STARTING,ACTUAL_RATE,PRED_P10,PRED_P50,PRED_P90,ABS_ERR,IN_80,ACTUAL_STAT,PRED_STAT_P10,PRED_STAT_P50,PRED_STAT_P90,STAT_ABS_ERR
25,Jonathan Kuminga,ATL,ATL @ NYK,34.733333,0,34.733333,15.023561,22.997015,30.752520,11.736318,False,34.733333,15.023561,22.997015,30.752520,11.736318
21,Jamal Shead,TOR,TOR @ CLE,37.600000,1,37.600000,20.514738,28.639973,34.976036,8.960027,False,37.600000,20.514738,28.639973,34.976036,8.960027
13,Dyson Daniels,ATL,ATL @ NYK,26.245000,1,26.245000,25.041609,34.610722,38.480858,8.365722,True,26.245000,25.041609,34.610722,38.480858,8.365722
9,Dean Wade,CLE,CLE vs. TOR,28.453333,1,28.453333,14.808070,20.505333,27.157488,7.948000,False,28.453333,14.808070,20.505333,27.157488,7.948000
0,Aaron Gordon,DEN,DEN vs. MIN,36.883333,1,36.883333,22.270609,29.227121,36.005070,7.656212,False,36.883333,22.270609,29.227121,36.005070,7.656212
32,Miles McBride,NYK,NYK vs. ATL,13.141667,0,13.141667,12.501974,20.280464,28.200087,7.138798,True,13.141667,12.501974,20.280464,28.200087,7.138798
24,Jaylon Tyson,CLE,CLE vs. TOR,11.733333,0,11.733333,11.591450,18.339474,25.440586,6.606140,True,11.733333,11.591450,18.339474,25.440586,6.606140
30,Max Strus,CLE,CLE vs. TOR,26.550000,0,26.550000,14.309460,20.193312,25.622223,6.356688,False,26.550000,14.309460,20.193312,25.622223,6.356688
8,Collin Murray-Boyles,TOR,TOR @ CLE,25.905000,0,25.905000,10.520421,19.589935,27.220566,6.315065,True,25.905000,10.520421,19.589935,27.220566,6.315065
16,Ja'Kobe Walter,TOR,TOR @ CLE,27.955000,0,27.955000,13.402122,21.715137,29.600044,6.239863,True,27.955000,13.402122,21.715137,29.600044,6.239863



Best hits


,PLAYER_NAME,TEAM_ABBREVIATION,MATCHUP,MIN,STARTING,ACTUAL_RATE,PRED_P10,PRED_P50,PRED_P90,ABS_ERR,IN_80,ACTUAL_STAT,PRED_STAT_P10,PRED_STAT_P50,PRED_STAT_P90,STAT_ABS_ERR
28,Julius Randle,MIN,MIN @ DEN,36.350000,1,36.350000,25.844301,36.071377,39.671680,0.278623,True,36.350000,25.844301,36.071377,39.671680,0.278623
3,Brandon Ingram,TOR,TOR @ CLE,35.895000,1,35.895000,26.379021,35.570744,39.609287,0.324256,True,35.895000,26.379021,35.570744,39.609287,0.324256
12,Donte DiVincenzo,MIN,MIN @ DEN,30.568333,1,30.568333,20.044596,30.148401,34.987980,0.419932,True,30.568333,20.044596,30.148401,34.987980,0.419932
42,Sandro Mamukelashvili,TOR,TOR @ CLE,20.511667,0,20.511667,13.954448,21.208355,29.190123,0.696688,True,20.511667,13.954448,21.208355,29.190123,0.696688
15,Gabe Vincent,ATL,ATL @ NYK,14.216667,0,14.216667,7.623916,14.947675,22.716993,0.731008,True,14.216667,7.623916,14.947675,22.716993,0.731008
29,Karl-Anthony Towns,NYK,NYK vs. ATL,33.775000,1,33.775000,25.097242,33.022667,37.714272,0.752333,True,33.775000,25.097242,33.022667,37.714272,0.752333
33,Mitchell Robinson,NYK,NYK vs. ATL,18.150000,0,18.150000,11.995686,17.239466,24.599545,0.910534,True,18.150000,11.995686,17.239466,24.599545,0.910534
19,Jalen Johnson,ATL,ATL @ NYK,35.633333,1,35.633333,28.010220,36.611259,41.720951,0.977926,True,35.633333,28.010220,36.611259,41.720951,0.977926
45,Tony Bradley,ATL,ATL @ NYK,12.033333,0,12.033333,5.985093,10.964952,19.710651,1.068382,True,12.033333,5.985093,10.964952,19.710651,1.068382
22,James Harden,CLE,CLE vs. TOR,34.883333,1,34.883333,27.520567,36.131115,40.099792,1.247782,True,34.883333,27.520567,36.131115,40.099792,1.247782
